## Catastrophic Forgetting

When a model is trained on new data, it can overwrite what it previously learned — not just adjust it, but lose it almost entirely.
This is the central risk of fine-tuning any pre-trained model.

## How it actually works

**Where knowledge lives in a neural network:**
Everything a model "knows" is encoded in its weights — millions or billions of numbers
adjusted during training to produce useful outputs.
There is no separate, labelled storage for "old knowledge" versus "new knowledge."
It is all the same set of numbers.

**What happens during fine-tuning:**
You take a pre-trained model and continue training it on new, narrower data.
Gradient descent adjusts the weights to reduce error on this new data.
But it has no built-in way to know which weights were important for OLD tasks
versus which ones are being adjusted for the NEW task.
It simply moves every weight in whatever direction reduces the new loss —
even if that direction destroys something useful from before.

**Why this happens more in narrow fine-tuning:**
If your new training data is narrow and repetitive — say, thousands of D365 invoice examples —
the model receives a strong, consistent pull in one direction.
That strong pull can overwrite weights that encoded broader, more general knowledge,
because nothing in the training process tells it to protect that general knowledge.

**The connection to vanishing/exploding gradients (notebook 23):**
Residual connections solve a TRAINING STABILITY problem — keeping gradients alive across depth.
Catastrophic forgetting is a different problem — DATA INTERFERENCE.
Even with perfectly stable training, a model can still forget previous knowledge
if the new training signal overwrites the relevant weights.b

## Where it breaks or gets dangerous

**The silent failure pattern:**
You fine-tune a model to be excellent at D365 invoice classification.
It now performs brilliantly on invoices. But ask it a general coding question,
or a question outside the fine-tuned domain, and quality has quietly dropped —
sometimes dramatically. Nobody told you this would happen. There is no warning.
You only discover it when a user complains the assistant "got worse."

**Compounding fine-tunes:**
You fine-tune once for invoices. Six months later, the client's vendor rules change,
so you fine-tune again on the updated rules. The SECOND fine-tune can partially
overwrite what the FIRST one taught the model. You now have a model that has
forgotten some of round one's lessons while learning round two's lessons.
Each round of fine-tuning is a fresh risk of forgetting something useful.

**Why this matters more for enterprise AI than for hobby projects:**
A hobbyist fine-tuning a model for a fun side project rarely notices subtle forgetting.
An enterprise deploying a fine-tuned model across multiple business functions
absolutely will — and the cost of that silent degradation is real:
wrong answers, lost trust, debugging time nobody budgeted for.

**Mitigations (the three main approaches):**
1. **LoRA / QLoRA** — Low-Rank Adaptation. Instead of updating ALL the model's weights,
   you freeze the original weights and train a small set of additional parameters
   that sit alongside them. The original knowledge stays intact because you
   never touched those weights at all. This is the dominant approach used today.
2. **Replay** — mix some of the ORIGINAL training data in with the new data
   during fine-tuning, so the model keeps seeing examples of what it should not forget.
3. **Elastic Weight Consolidation (EWC)** — a more advanced technique that identifies
   which weights were most important for previous tasks and adds a penalty
   that discourages changing those specific weights too much.

In [ ]:
# Catastrophic Forgetting — conceptual illustration

import numpy as np

class SimpleModel:
    """A toy model with weights representing 'knowledge'."""
    def __init__(self, num_weights=10):
        # Each weight represents some piece of learned knowledge
        self.weights = np.random.uniform(0.4, 0.6, num_weights)

    def predict(self, task_relevant_indices):
        """Simulate performance on a task using only relevant weights."""
        return self.weights[task_relevant_indices].mean()

def fine_tune_naive(model, task_indices, target_value, steps=20, lr=0.3):
    """
    Naive fine-tuning: adjusts ALL weights toward the target,
    with no protection for weights important to other tasks.
    """
    for _ in range(steps):
        current = model.predict(task_indices)
        error = target_value - current
        # Update ALL weights, not just task-relevant ones — this is the problem
        model.weights += lr * error * 0.1
    return model

def fine_tune_with_lora_style_protection(model, task_indices, target_value, steps=20, lr=0.3):
    """
    LoRA-style approach: only adjust weights relevant to THIS task,
    leaving other weights completely frozen.
    """
    for _ in range(steps):
        current = model.predict(task_indices)
        error = target_value - current
        # Only update the task-relevant weights — everything else stays frozen
        model.weights[task_indices] += lr * error * 0.3
    return model

# Set up two "tasks" the model needs to handle
np.random.seed(7)
original_model = SimpleModel(num_weights=10)

task_A_indices = np.array([0, 1, 2])   # e.g. "general coding knowledge"
task_B_indices = np.array([7, 8, 9])   # e.g. "D365 invoice classification"

print("=== BEFORE any fine-tuning ===")
print(f"Task A (general knowledge) performance: {original_model.predict(task_A_indices):.3f}")
print(f"Task B (D365 invoices) performance:     {original_model.predict(task_B_indices):.3f}")

print()
print("=== NAIVE FULL FINE-TUNING for Task B ===")
naive_model = SimpleModel(num_weights=10)
naive_model.weights = original_model.weights.copy()
naive_model = fine_tune_naive(naive_model, task_B_indices, target_value=0.95)
print(f"Task A performance AFTER fine-tuning Task B: {naive_model.predict(task_A_indices):.3f}  <-- dropped, even though we never meant to touch Task A")
print(f"Task B performance AFTER fine-tuning:          {naive_model.predict(task_B_indices):.3f}")

print()
print("=== LoRA-STYLE PROTECTED FINE-TUNING for Task B ===")
protected_model = SimpleModel(num_weights=10)
protected_model.weights = original_model.weights.copy()
protected_model = fine_tune_with_lora_style_protection(protected_model, task_B_indices, target_value=0.95)
print(f"Task A performance AFTER fine-tuning Task B: {protected_model.predict(task_A_indices):.3f}  <-- unchanged, because those weights were frozen")
print(f"Task B performance AFTER fine-tuning:          {protected_model.predict(task_B_indices):.3f}")

print()
print("Naive fine-tuning improved Task B but quietly damaged Task A.")
print("LoRA-style fine-tuning improved Task B while leaving Task A completely intact.")

## My D365 analogy

Catastrophic forgetting is like a consultant doing a deep, intensive override of a client's
posting profiles during an urgent fix, without checking what else those same shared
parameters control elsewhere in the system.

The urgent fix gets resolved. The specific problem is solved.
But three weeks later, someone discovers that another unrelated process
quietly broke, because it depended on a setting that got changed
during that "unrelated" fix — and nobody checked the blast radius first.

The fix touched shared infrastructure without protecting what else relied on it.

LoRA is the equivalent of making a CONTROLLED, ISOLATED change —
adding a new layer of configuration specifically for the new requirement,
without touching the underlying shared parameters that everything else depends on.
The original setup stays completely intact. Only the new, additive layer changes.

This is exactly the discipline every senior consultant learns the hard way:
never assume a fix is isolated just because it solves the problem in front of you.
Check what else depends on the thing you are about to change.

## LinkedIn post idea

You fine-tune a model to be brilliant at one specific task.
A few weeks later, someone notices it has quietly gotten worse at something else —
something nobody touched, something that should have been completely unrelated.

This is catastrophic forgetting. Training on new data does not add knowledge
on top of old knowledge. It can overwrite it, silently, with no warning.

The fix that the field landed on is almost elegant in its restraint:
stop touching the original weights at all. Freeze them completely.
Train a small, separate set of new parameters that sit alongside the frozen original —
adjusting behaviour for the new task without ever risking the old one.

It is called LoRA — Low-Rank Adaptation — and it is now the standard way
serious teams fine-tune models in production.

The lesson is bigger than the technique. Every system that gets modified repeatedly
over time — a model, a D365 configuration, a piece of shared infrastructure —
eventually needs the same discipline: change what you must, in isolation,
and leave everything you are not actively improving completely untouched.